# Load packages

In [ ]:
import os
import ee
import geemap
import json
import requests
import folium

from IPython.display import display, HTML, clear_output

# Initialise GEE

You will need to authenticate your Google account the first time you run this. 

In [ ]:
# Initialize Earth Engine
try:
    ee.Initialize()
    print("Earth Engine already initialized")
except Exception as e:
    ee.Authenticate()
    ee.Initialize()
    print("Earth Engine initialized")

## Plot the basemap

The first time you run the code below, follow the link to authenticate your Google account. After following the instructions online, you will receive an authorization code, which you can paste back into the input box in your notebook. If you're using VSCode, this will appear at the top of your window where the search bar is.

In [ ]:
Map = geemap.Map(lite_mode=True, zoom=2)
Map.add_basemap("SATELLITE")
# Map

# Clean up existing HTML files
html_files = ['satellite_map.html', 
              'satellite_map2.html', 
              'satellite_map3.html',
              'satellite_map4.html',
              'satellite_map_S2.html']
for file in html_files:
    if os.path.exists(file):
        os.remove(file)

# Clear previous outputs before displaying new map
clear_output(wait=True)
Map.to_html(filename='satellite_map.html')
HTML('satellite_map.html')

## Generate AOI

There are several approaches to create areas of interest (AOIs).

### Manually create an AOI from coordinates

In [ ]:
# Alternative: Define AOI from coordinates
# aoi = ee.Geometry.Rectangle([-122.5, 37.5, -122.0, 38.0])  # San Francisco area

### Function to read an AOI from file

### Function for multiple features in the json 

This function allows for filtering to just select certain features. Here we just want the 'North Australian Tropical Savanna'.

In [ ]:
def load_aoi_from_file(json_file_path, filter_field=None, filter_value=None):
    with open(json_file_path) as f:
        geojson = json.load(f)
    
    gtype = geojson.get('type')
    if gtype == 'FeatureCollection':
        features = [ee.Feature(feat) for feat in geojson['features']]
        fc = ee.FeatureCollection(features)
        if filter_field is not None:
            fc = fc.filter(ee.Filter.eq(filter_field, filter_value))
        return fc.geometry()
    elif gtype == 'Feature':
        return ee.Geometry(geojson['geometry'])
    else:
        return ee.Geometry(geojson)

### Load and plot the Tropical Savanna AOI

In [ ]:
# Load AOI from file
aoi_name = 'north_aus_tropical_savanna_w-buffer'

aoi = load_aoi_from_file(
    f'AOIs/{aoi_name}.geojson'
)

Map2 = geemap.Map(lite_mode=True)
Map2.add_basemap("SATELLITE")
# Map2.addLayer(aoi, {}, 'Area of Interest')
# add the json directly (not the one on the GEE server-side)
Map2.add_geojson(f'AOIs/{aoi_name}.geojson', layer_name='Area of Interest')
Map2.set_center(133.0, -15.0, 5)  # lon, lat, zoom — roughly central North Australia
# Map2

# Clear previous outputs before displaying new map
clear_output(wait=True)
Map2.to_html(filename='satellite_map2.html')
HTML('satellite_map2.html')

## Randomly sample AOIs

Because we want the training data to cover a large area that we will predict over, we can randomly sample images from across the prediction extent.

In [ ]:
half_side_m = 12500  # half the side length in metres

# sample N random points within the shrunken region
n_points = 100
random_points = ee.FeatureCollection.randomPoints(
    region=aoi,
    points=n_points,
    seed=202607082,
)

# turn each point into a rectangular AOI
def point_to_rect(feature):
    coords = feature.geometry().coordinates()  # [lon, lat] of the original point
    return feature.setGeometry(
        feature.geometry().buffer(half_side_m).bounds()
    ).set({
        'lon': coords.get(0),
        'lat': coords.get(1),
    })

rect_fc = random_points.map(point_to_rect)

### Plot the sampled AOIs

In [ ]:
Map3 = geemap.Map(lite_mode=True)
Map3.add_basemap("SATELLITE")
# Map3.addLayer(aoi, {}, 'Area of Interest')
# add the json directly (not the one on the GEE server-side)
Map3.add_geojson(f'AOIs/{aoi_name}.geojson', layer_name='Area of Interest')
Map3.set_center(133.0, -15.0, 5)  # lon, lat, zoom — roughly central North Australia
# Map3

# Style: red outlines, no fill
Map3.addLayer(
    rect_fc.style(color='red', fillColor='FF000033', width=2),  # last 2 hex digits = alpha
    {},
    'Sampled AOIs',
)

# Map3

# Clear previous outputs before displaying new map
clear_output(wait=True)
Map3.to_html(filename='satellite_map3.html')
HTML('satellite_map3.html')

# Get Sentinel-2 Collection with Cloud Masking

This function creates a cloud mask for Sentinel-2 imagery.

There are other approaches to filter clouds, such as the approach listed here: [https://developers.google.com/earth-engine/tutorials/community/sentinel-2-s2cloudless](https://developers.google.com/earth-engine/tutorials/community/sentinel-2-s2cloudless).

In [ ]:
def maskS2clouds_CSPlus(image):
    """
    Mask Sentinel-2 using Cloud Score+.
    Assumes the CS+ bands ('cs', 'cs_cdf') have already been linked
    to the S2 collection via linkCollection().
    """
    # Use the cs_cdf band (cumulative distribution function variant)
    # is generally more robust than 'cs' for time-series work.
    # Threshold range: 0 (not clear) to 1 (clear).
    #   0.60 = permissive (more pixels kept, some haze/thin cloud)
    #   0.65 = balanced (Google's commonly recommended default)
    #   0.80+ = strict (clean composites, fewer observations)
    QA_BAND = 'cs_cdf'
    CLEAR_THRESHOLD = 0.65

    mask = image.select(QA_BAND).gte(CLEAR_THRESHOLD)
    return (image.divide(10000)
                 .updateMask(mask)
                 .copyProperties(image, ['system:time_start']))

# Get Monthly Composites of Sentinel-2

## Define a function to get monthly composites

We want to create monthly composites of Sentinel-2 imagery. This function will filter the Sentinel-2 image collection by date and area of interest (AOI), apply the cloud mask, and then compute the median of each 'cloud-free' pixel for the month.

In [ ]:
def get_monthly_composites(start_date, end_date, aoi):
    """Generate monthly S2 composites using Cloud Score+ for cloud/shadow masking."""
    start = ee.Date(start_date)
    end = ee.Date(end_date)
    months = end.difference(start, 'month').round().int()

    # Load harmonized S2 SR and link Cloud Score+ bands once.
    # Pre-filter by AOI, date, and a generous cloud cover cap to drop the worst scenes
    # before the join.
    s2_base = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                 .filterBounds(aoi)
                 .filterDate(start_date, end_date)
                 .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 90)))

    csPlus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
    s2_linked = s2_base.linkCollection(csPlus, ['cs', 'cs_cdf'])

    raw_count = s2_base.size().getInfo()
    print(f"  Found {raw_count} S2 images for {start_date} to {end_date}")

    def get_monthly_image(month_index):
        current_month_start = start.advance(month_index, 'month')
        current_month_end = current_month_start.advance(1, 'month')
        date_format = current_month_start.format('YYYY-MM')

        # Filter the already-linked collection to this month and apply CS+ mask
        monthly = (s2_linked
                     .filterDate(current_month_start, current_month_end)
                     .map(maskS2clouds_CSPlus))

        composite = monthly.median().clip(aoi)

        # Bands are already scaled to 0–1 by maskS2clouds_CSPlus
        rgb = composite.select(['B2', 'B3', 'B4'])

        return rgb.set({
            'system:index': date_format,
            'system:time_start': current_month_start.millis()
        })

    month_indices = ee.List.sequence(0, months.subtract(1))
    composites = ee.ImageCollection.fromImages(month_indices.map(get_monthly_image))
    return composites

## Instead, get a temporal range longer than a month and take the median

In [ ]:
def get_period_composite(start_date, end_date, aoi):
    """Generate a single S2 composite across the whole period, using Cloud Score+."""
    s2_base = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                 .filterBounds(aoi)
                 .filterDate(start_date, end_date)
                 .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 90)))

    csPlus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
    s2_linked = s2_base.linkCollection(csPlus, ['cs', 'cs_cdf'])

    raw_count = s2_base.size().getInfo()
    print(f"  Found {raw_count} S2 images for {start_date} to {end_date}")

    masked = s2_linked.map(maskS2clouds_CSPlus)
    composite = masked.median().clip(aoi)

    rgb = composite.select(['B2', 'B3', 'B4'])

    period_id = f'{start_date}_to_{end_date}'
    return rgb.set({
        'system:index': period_id,
        'system:time_start': ee.Date(start_date).millis()
    })

## Define a function to export and download images

If the images are small enough, you can download them directly to your local machine. If they are too large, you can export them to your Google Drive.

In [ ]:
# For larger or higher resolution images, use Google Drive export:
def export_to_drive(composites, aoi, aoi_name, drive_folder='Earth_Engine_Exports', scale=10):
    image_list = composites.toList(composites.size())
    num_images = image_list.size().getInfo()
    tasks = []

    for i in range(num_images):
        image = ee.Image(image_list.get(i))
        date_str = image.get('system:index').getInfo()

        task = ee.batch.Export.image.toDrive(
            image=image.visualize(min=0, max=0.3, bands=['B4', 'B3', 'B2']),
            description=f'{aoi_name}_{date_str}',
            folder=drive_folder,
            fileNamePrefix=f'{aoi_name}_{date_str}',
            scale=scale,
            region=aoi,
            crs='EPSG:4326',
            maxPixels=1e10,
        )
        task.start()
        tasks.append(task)
        print(f"Started: {aoi_name}_{date_str}")

    return tasks  # return so you can monitor with task.status()

# Download images for a specified date range

We will get an image for June 2024 as an example.

In [ ]:
# Define parameters
start_date = '2024-02-01'
end_date = '2024-08-31'
output_folder = 'image/sentinel2_monthly_images'

### Run the function

The function will print how many suitable images were found for the specified date range and AOI. If no images are found, you may need to adjust your date range or AOI.

In [ ]:
# Grab the first randomly sampled AOI's geometry
single_aoi = ee.Feature(rect_fc.first()).geometry()

# Get monthly composites
# composites = get_monthly_composites(start_date, end_date, single_aoi)
composites = ee.ImageCollection([get_period_composite(start_date, end_date, single_aoi)])

## Plot one of the randomly sampled AOIs

In [ ]:
# Visualize a preview
Map4 = geemap.Map()
Map4.centerObject(single_aoi, zoom=12)
Map4.add_basemap("SATELLITE")
Map4.addLayer(single_aoi, {}, 'Area of Interest')

# Add the first image to the map
first_image = ee.Image(composites.first())
viz_params = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.25}
# Map4.addLayer(first_image, viz_params, 'First Monthly Composite')
# Map4

# Get the number of images in the collection
count = composites.size().getInfo()
print(f"Found {count} monthly composites")

# Add each monthly composite to the map with its date as the layer name
if count > 0:
    # Convert to list for iterating
    date_strs = composites.aggregate_array('system:index').getInfo()
    image_list = composites.toList(count)
    
    # Add each image as a separate layer
    for i, date_str in enumerate(date_strs):
        image = ee.Image(image_list.get(i))
        Map4.addLayer(image, viz_params, f'S2 RGB {date_str}', shown=(i == 0))
           
    print(f"Added {count} layers to the map. Use the layer control to toggle visibility.")
else:
    print("No composites available to display.")

# Add the layer control (if not already visible)
Map4.add_layer_control()

# Display the map
# Map4

# Clear previous outputs before displaying new map
clear_output(wait=True)
Map4.to_html(filename='satellite_map4.html')
HTML('satellite_map4.html')

# Export images to Google Drive

## Set up parameters for exporting the images

In [ ]:
# Convert the FeatureCollection to a list and get the count
rect_list = rect_fc.toList(rect_fc.size())
n = rect_fc.size().getInfo()

lons = rect_fc.aggregate_array('lon').getInfo()
lats = rect_fc.aggregate_array('lat').getInfo()

def coord_tag(lat, lon):
    """e.g. lat=-42.8826, lon=147.3257 -> 'S42p88_E147p33'"""
    ns = 'S' if lat < 0 else 'N'
    ew = 'W' if lon < 0 else 'E'
    return f"{ns}{abs(lat):.2f}_{ew}{abs(lon):.2f}".replace('.', 'p')


# base_name = aoi_name  # preserve the original
base_name = "north_aus_tropical_savanna"

## Export images

In [ ]:
print(f'Processing {n} AOIs...')

all_tasks = []

for i in range(n):
    single_aoi = ee.Feature(rect_list.get(i)).geometry()
    tag = coord_tag(lats[i], lons[i])
    single_name = f'{base_name}_{i:03d}_{tag}'   
    
    print(f'\n[{i+1}/{n}] AOI: {single_name}')

    try:
        # composites = get_monthly_composites(start_date, end_date, single_aoi)
        composites = ee.ImageCollection([get_period_composite(start_date, end_date, single_aoi)])
        tasks = export_to_drive(
            composites,
            aoi=single_aoi,
            aoi_name=single_name,
            drive_folder='sentinel2_images',
            scale=10,
        )
        all_tasks.extend(tasks)
    except Exception as e:
        print(f'  FAILED for {single_name}: {e}')
        continue

print(f'\nStarted {len(all_tasks)} total export tasks.')

### To check the status of the downloads (from the server-side)

In [ ]:
from collections import Counter
states = Counter(t.status()['state'] for t in all_tasks)
print(states)   

### If you need to kill all tasks

In [ ]:
# for t in all_tasks:
#     if t.status()['state'] in ('READY', 'RUNNING'):
#         t.cancel()